# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and includes protein abundance, modification analysis, peptide sequences, and related fields for extracellular vesicles from human mast cells.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant metadata URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata and print main information
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, their `@id`s and basic metadata.

In [ ]:
# List all record sets by @id, name, and description
print("Available Record Sets:")
for rs in metadata.record_sets:
    print(f"@id: {rs['@id']}, name: {rs['name'] if 'name' in rs else '(no name)'}")
    if 'description' in rs:
        print(f"    Description: {rs['description']}")
    print(f"    Fields (@ids): {[f['@id'] for f in rs['fields']]}")

# Save the list of record set @id's for later
record_set_ids = [rs['@id'] for rs in metadata.record_sets]

## 3. Data Extraction
Load data from each record set into a pandas DataFrame. You can change the record set or field by specifying the corresponding `@id`.

In [ ]:
# Extract data from each record set (@id) into dataframes
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for record set: {record_set_id}")

# Print all columns available from the first record set
example_record_set_id = record_set_ids[0]
print(f"\nColumns for record set {example_record_set_id}:\n{dataframes[example_record_set_id].columns.tolist()}")
# Show a preview
dataframes[example_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Let's perform some basic data filtering, normalization and grouping. We'll operate on the first record set for demonstration.

In [ ]:
# Replace with a numeric field @id from the example record set
example_df = dataframes[example_record_set_id]

# Show all available columns (often @id or field labels)
print(example_df.columns.tolist())

# Let's try a common mass-spec field like abundance, coverage or peptide count
# Replace this with the correct column name if available (see print above)
numeric_field = None
for candidate in ['abundance', 'normalized_abundance', 'coverage', 'peptide_count', 'MW', 'pI', 'molecular_weight']:
    if candidate in example_df.columns:
        numeric_field = candidate
        break

if numeric_field is None:
    # Try to pick the first float/integer column by type
    numeric_cols = example_df.select_dtypes(include=['number']).columns
    if len(numeric_cols) > 0:
        numeric_field = numeric_cols[0]

print(f"Using numeric field: {numeric_field}")

# Filter records with value > threshold
threshold = 10
if numeric_field is not None:
    filtered_df = example_df[example_df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize numeric field
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()

    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Attempt to group by a likely categorical field
    group_field = None
    for candidate in ['sample', 'condition', 'group', 'description', 'accession', 'protein_id']:
        if candidate in filtered_df.columns and candidate != numeric_field:
            group_field = candidate
            break

    if group_field is not None:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"\nGrouped data by {group_field}:")
        print(grouped_df.head())
    else:
        print("No categorical field found to group by.")
else:
    print("No numeric field available for EDA analysis.")

## 5. Visualization
Visualize the distribution of the selected numeric field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None and numeric_field in example_df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(example_df[numeric_field].dropna(), bins=30)
    plt.xlabel(numeric_field)
    plt.title(f"Distribution of {numeric_field}")
    plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion

This notebook demonstrated how to use `mlcroissant` to explore the FAIR² mass spectrometry dataset, including how to load its metadata and records, review available fields by their `@id`, extract structured data, and perform basic EDA and visualization. For further or domain-specific analyses, refer to the dataset's schema and documentation, and adjust field selections accordingly.